In [0]:
%sql
TRUNCATE TABLE _exponent.omop_tw.device_exposure;

In [0]:
%sql
DELETE FROM _exponent.omop_silver.device_exposure
WHERE source_system = 'allscripts_tw';

In [0]:
%sql
DELETE FROM _exponent.omop_mapping.source_to_device_exposure
WHERE source_system = 'allscripts_tw';

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW device_mapping AS
WITH concept_relationship_maps_to_device_dedup AS (
  SELECT
    _exponent.omop.concept_relationship.concept_id_1 AS source_concept_id,
    _exponent.omop.concept_relationship.concept_id_2 AS standard_concept_id,
    ROW_NUMBER() OVER (
      PARTITION BY _exponent.omop.concept_relationship.concept_id_1
      ORDER BY _exponent.omop.concept_relationship.concept_id_2 ASC
    ) AS row_number_within_source
  FROM _exponent.omop.concept_relationship
  INNER JOIN _exponent.omop.concept AS target_concept
    ON target_concept.concept_id =
       _exponent.omop.concept_relationship.concept_id_2
   AND target_concept.standard_concept = 'S'
   AND target_concept.invalid_reason IS NULL
   AND target_concept.domain_id = 'Device'
  WHERE _exponent.omop.concept_relationship.relationship_id = 'Maps to'
    AND _exponent.omop.concept_relationship.invalid_reason IS NULL
)
SELECT
  _exponent.omop.concept.concept_id AS source_concept_id,
  _exponent.omop.concept.concept_code AS source_concept_code,
  _exponent.omop.concept.vocabulary_id,
  _exponent.omop.concept.domain_id,
  _exponent.omop.concept.concept_class_id,
  concept_relationship_maps_to_device_dedup.standard_concept_id
FROM _exponent.omop.concept
LEFT JOIN concept_relationship_maps_to_device_dedup
  ON concept_relationship_maps_to_device_dedup.source_concept_id =
     _exponent.omop.concept.concept_id
 AND concept_relationship_maps_to_device_dedup.row_number_within_source = 1
WHERE _exponent.omop.concept.vocabulary_id IN ('HCPCS', 'SNOMED')
  AND _exponent.omop.concept.domain_id = 'Device'
  AND _exponent.omop.concept.invalid_reason IS NULL;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW device_exposure_charge AS
WITH charge_dates_cte AS (
  SELECT
    dbo_charge.id,

    CASE
      WHEN dbo_charge.postdttm IS NULL THEN NULL
      WHEN CAST(dbo_charge.postdttm AS STRING) = '-' THEN NULL
      WHEN CAST(dbo_charge.postdttm AS TIMESTAMP) < TIMESTAMP('1970-01-01') THEN NULL
      ELSE CAST(dbo_charge.postdttm AS TIMESTAMP)
    END AS postdttm,

    CASE
      WHEN dbo_charge.starttime IS NULL THEN NULL
      WHEN CAST(dbo_charge.starttime AS STRING) = '-' THEN NULL
      WHEN CAST(dbo_charge.starttime AS TIMESTAMP) < TIMESTAMP('1970-01-01') THEN NULL
      ELSE CAST(dbo_charge.starttime AS TIMESTAMP)
    END AS starttime,

    CASE
      WHEN dbo_charge.endtime IS NULL THEN NULL
      WHEN CAST(dbo_charge.endtime AS STRING) = '-' THEN NULL
      WHEN CAST(dbo_charge.endtime AS TIMESTAMP) < TIMESTAMP('1970-01-01') THEN NULL
      ELSE CAST(dbo_charge.endtime AS TIMESTAMP)
    END AS endtime

  FROM _exponent._bronze_allscripts_tw_works_vw.dbo_charge
)

SELECT DISTINCT
  CONCAT_WS(
    chr(31),
    'allscripts_tw',
    'dbo_charge',
    'id',
    CAST(dbo_charge.id AS BIGINT)
  ) AS device_exposure_source_value,

  source_to_person.person_id AS person_id,

  COALESCE(device_mapping.standard_concept_id, 0) AS device_concept_id,

  CAST(
    COALESCE(
      charge_dates_cte.starttime,
      charge_dates_cte.endtime,
      charge_dates_cte.postdttm
    ) AS DATE
  ) AS device_exposure_start_date,

  COALESCE(
    charge_dates_cte.starttime,
    charge_dates_cte.endtime,
    charge_dates_cte.postdttm
  ) AS device_exposure_start_datetime,

  CAST(
    COALESCE(
      charge_dates_cte.endtime,
      charge_dates_cte.starttime,
      charge_dates_cte.postdttm
    ) AS DATE
  ) AS device_exposure_end_date,

  COALESCE(
    charge_dates_cte.endtime,
    charge_dates_cte.starttime,
    charge_dates_cte.postdttm
  ) AS device_exposure_end_datetime,

  CAST(32817 AS INT) AS device_type_concept_id, -- TODO: confirm best OMOP type concept
  CAST(NULL AS STRING) AS unique_device_id,
  NULL AS production_id,
  CAST(COALESCE(dbo_charge.unitstobillfor, 1) AS DOUBLE) AS quantity,
  source_to_provider.provider_id AS provider_id,
  source_to_visit_occurrence.visit_occurrence_id AS visit_occurrence_id,
  CAST(NULL AS BIGINT) AS visit_detail_id,

  COALESCE(
    NULLIF(REGEXP_REPLACE(TRIM(dbo_charge_code_de.cpt4code), '[\\s\\u00A0]+', ''), ''),
    NULLIF(REGEXP_REPLACE(TRIM(dbo_charge_code_de.entrycode), '[\\s\\u00A0]+', ''), '')
  ) AS device_source_value,

  COALESCE(device_mapping.source_concept_id, 0) AS device_source_concept_id,

  NULL AS unit_concept_id,
  NULL AS unit_source_value,
  NULL AS unit_source_concept_id,

  'allscripts_tw' AS source_system,

  CONCAT_WS(
    '|',
    CAST(COALESCE(source_to_person.person_id, 0) AS STRING),
    CAST(COALESCE(device_mapping.standard_concept_id, 0) AS STRING),
    CAST(
      COALESCE(
        charge_dates_cte.starttime,
        charge_dates_cte.endtime,
        charge_dates_cte.postdttm
      ) AS STRING
    ),
    CAST(COALESCE(source_to_provider.provider_id, 0) AS STRING),
    CAST(COALESCE(source_to_visit_occurrence.visit_occurrence_id, 0) AS STRING),
    COALESCE(
      NULLIF(REGEXP_REPLACE(TRIM(dbo_charge_code_de.cpt4code), '[\\s\\u00A0]+', ''), ''),
      NULLIF(REGEXP_REPLACE(TRIM(dbo_charge_code_de.entrycode), '[\\s\\u00A0]+', ''), '')
    )
  ) AS dedupe_key,

  CAST(2 AS INT) AS source_priority

FROM _exponent._bronze_allscripts_tw_works_vw.dbo_charge AS dbo_charge

INNER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_charge_code_de AS dbo_charge_code_de
  ON dbo_charge_code_de.id = dbo_charge.chargecodede

INNER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_visit AS dbo_visit
  ON dbo_visit.id = dbo_charge.visitid

LEFT JOIN charge_dates_cte
  ON charge_dates_cte.id = dbo_charge.id

JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_person AS dbo_person
  ON dbo_person.id = CAST(dbo_visit.patientid AS BIGINT)


LEFT JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_provider AS dbo_provider
  ON dbo_provider.id = CAST(COALESCE(dbo_charge.billingproviderid, dbo_charge.otherproviderid) AS BIGINT)

LEFT JOIN device_mapping AS device_mapping
  ON device_mapping.source_concept_code = COALESCE(
       NULLIF(REGEXP_REPLACE(TRIM(dbo_charge_code_de.cpt4code), '[\\s\\u00A0]+', ''), ''),
       NULLIF(REGEXP_REPLACE(TRIM(dbo_charge_code_de.entrycode), '[\\s\\u00A0]+', ''), '')
     )

JOIN _exponent.omop_mapping.source_to_person AS source_to_person
  ON source_to_person.person_source_value = CONCAT_WS(
       chr(31),
       'allscripts_tw',
       'dbo_person',
       'id',
       CAST(dbo_person.id AS BIGINT)
     )
 AND source_to_person.active_flag = TRUE

LEFT JOIN _exponent.omop_mapping.source_to_provider AS source_to_provider
  ON source_to_provider.provider_source_value = CONCAT_WS(
       chr(31),
       'allscripts_tw',
       'dbo_provider',
       'id',
       CAST(dbo_provider.id AS BIGINT)
     )
 AND source_to_provider.active_flag = TRUE

LEFT JOIN _exponent.omop_mapping.source_to_visit_occurrence AS source_to_visit_occurrence
  ON source_to_visit_occurrence.visit_occurrence_source_value = CONCAT_WS(
       chr(31),
       'allscripts_tw',
       'dbo_visit',
       'id',
       CAST(dbo_visit.id AS BIGINT)
     )
 AND source_to_visit_occurrence.active_flag = TRUE

WHERE dbo_charge.id IS NOT NULL
  AND dbo_visit.patientid IS NOT NULL
  AND source_to_person.person_id IS NOT NULL
  AND COALESCE(
        NULLIF(REGEXP_REPLACE(TRIM(dbo_charge_code_de.cpt4code), '[\\s\\u00A0]+', ''), ''),
        NULLIF(REGEXP_REPLACE(TRIM(dbo_charge_code_de.entrycode), '[\\s\\u00A0]+', ''), '')
      ) IS NOT NULL
  AND COALESCE(device_mapping.standard_concept_id, 0) <> 0
  AND COALESCE(
        charge_dates_cte.starttime,
        charge_dates_cte.endtime,
        charge_dates_cte.postdttm
      ) IS NOT NULL
  AND dbo_charge.etl_load_ts BETWEEN CURRENT_TIMESTAMP() - INTERVAL 365 DAYS AND CURRENT_TIMESTAMP();

In [0]:
%sql
/*Commenting out for now.  Devices do not seem to show up under order activity.*/
-- CREATE OR REPLACE TEMPORARY VIEW device_exposure_activity AS
-- WITH cte_order_activity_ranked AS (
--   SELECT
--     dbo_order_activity.*,
--     ROW_NUMBER() OVER (
--       PARTITION BY dbo_order_activity.ordernumberext
--       ORDER BY
--         dbo_order_activity.createddttm DESC,
--         dbo_order_activity.orderactivityid DESC
--     ) AS row_number
--   FROM _exponent._bronze_allscripts_tw_works_vw.dbo_order_activity AS dbo_order_activity
-- ),

-- cte_order_activity AS (
--   SELECT
--     cte_order_activity_ranked.*
--   FROM cte_order_activity_ranked
--   WHERE cte_order_activity_ranked.row_number = 1
-- ),

-- activity_dates_cte AS (
--   SELECT
--     dbo_item_result.id AS item_result_id,

--     CASE
--       WHEN dbo_item_result.performeddttm IS NULL THEN NULL
--       WHEN CAST(dbo_item_result.performeddttm AS STRING) = '-' THEN NULL
--       WHEN CAST(dbo_item_result.performeddttm AS TIMESTAMP) < TIMESTAMP('1970-01-01') THEN NULL
--       ELSE CAST(dbo_item_result.performeddttm AS TIMESTAMP)
--     END AS performeddttm,

--     CASE
--       WHEN dbo_order_activity_header.createdttm IS NULL THEN NULL
--       WHEN CAST(dbo_order_activity_header.createdttm AS STRING) = '-' THEN NULL
--       WHEN CAST(dbo_order_activity_header.createdttm AS TIMESTAMP) < TIMESTAMP('1970-01-01') THEN NULL
--       ELSE CAST(dbo_order_activity_header.createdttm AS TIMESTAMP)
--     END AS createdttm

--   FROM _exponent._bronze_allscripts_tw_works_vw.dbo_order_activity_header AS dbo_order_activity_header
--   INNER JOIN cte_order_activity AS dbo_order_activity
--     ON dbo_order_activity.orderactivityheaderid = dbo_order_activity_header.id
--   INNER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_encounter AS dbo_encounter
--     ON dbo_encounter.id = dbo_order_activity_header.encounterid
--   INNER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_item_result AS dbo_item_result
--     ON dbo_item_result.orderitemext = dbo_order_activity.ordernumberext
--    AND dbo_item_result.patientid = dbo_encounter.patientid
-- )

-- SELECT DISTINCT
--   CONCAT_WS(
--     chr(31),
--     'allscripts_tw',
--     'dbo_item_result',
--     'id',
--     CAST(dbo_item_result.id AS BIGINT)
--   ) AS device_exposure_source_value,

--   source_to_person.person_id AS person_id,

--   COALESCE(device_mapping.standard_concept_id, 0) AS device_concept_id,

--   CAST(
--     COALESCE(
--       activity_dates_cte.performeddttm,
--       activity_dates_cte.createdttm
--     ) AS DATE
--   ) AS device_exposure_start_date,

--   COALESCE(
--     activity_dates_cte.performeddttm,
--     activity_dates_cte.createdttm
--   ) AS device_exposure_start_datetime,

--   CAST(
--     COALESCE(
--       activity_dates_cte.performeddttm,
--       activity_dates_cte.createdttm
--     ) AS DATE
--   ) AS device_exposure_end_date,

--   COALESCE(
--     activity_dates_cte.performeddttm,
--     activity_dates_cte.createdttm
--   ) AS device_exposure_end_datetime,

--   CAST(32817 AS INT) AS device_type_concept_id, -- TODO: confirm best OMOP type concept
--   CAST(NULL AS STRING) AS unique_device_id,
--   CAST(1 AS DOUBLE) AS quantity,
--   source_to_provider.provider_id AS provider_id,
--   source_to_visit_occurrence.visit_occurrence_id AS visit_occurrence_id,
--   CAST(NULL AS BIGINT) AS visit_detail_id,

--   COALESCE(
--     NULLIF(REGEXP_REPLACE(dbo_qo_classification_de.hicpcscode, '[\\s\\u00A0]+', ''), ''),
--     NULLIF(REGEXP_REPLACE(dbo_qo_classification_de.cpt4code, '[\\s\\u00A0]+', ''), '')
--   ) AS device_source_value,

--   COALESCE(device_mapping.source_concept_id, 0) AS device_source_concept_id,

--   'allscripts_tw' AS source_system,

--   CONCAT_WS(
--     '|',
--     CAST(COALESCE(source_to_person.person_id, 0) AS STRING),
--     CAST(COALESCE(device_mapping.standard_concept_id, 0) AS STRING),
--     CAST(
--       COALESCE(
--         activity_dates_cte.performeddttm,
--         activity_dates_cte.createdttm
--       ) AS STRING
--     ),
--     CAST(COALESCE(source_to_provider.provider_id, 0) AS STRING),
--     CAST(COALESCE(source_to_visit_occurrence.visit_occurrence_id, 0) AS STRING),
--     COALESCE(
--       NULLIF(REGEXP_REPLACE(dbo_qo_classification_de.hicpcscode, '[\\s\\u00A0]+', ''), ''),
--       NULLIF(REGEXP_REPLACE(dbo_qo_classification_de.cpt4code, '[\\s\\u00A0]+', ''), '')
--     )
--   ) AS dedupe_key,

--   CAST(1 AS INT) AS source_priority

-- FROM _exponent._bronze_allscripts_tw_works_vw.dbo_order_activity_header AS dbo_order_activity_header

-- INNER JOIN cte_order_activity AS dbo_order_activity
--   ON dbo_order_activity.orderactivityheaderid = dbo_order_activity_header.id

-- INNER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_encounter
--   ON dbo_encounter.id = dbo_order_activity_header.encounterid

-- INNER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_item_result
--   ON dbo_item_result.orderitemext = dbo_order_activity.ordernumberext
--  AND dbo_item_result.patientid = dbo_encounter.patientid

-- INNER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_qo_classification_de
--   ON dbo_qo_classification_de.id = dbo_item_result.qoclassificationde

-- LEFT JOIN activity_dates_cte
--   ON activity_dates_cte.item_result_id = dbo_item_result.id

-- LEFT JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_visit AS dbo_visit
--   ON dbo_visit.id = dbo_encounter.visitid

-- LEFT JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_person AS dbo_person
--   ON dbo_person.id = CAST(dbo_encounter.patientid AS BIGINT)

-- LEFT JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_provider AS dbo_provider
--   ON dbo_provider.id = CAST(dbo_order_activity.orderingproviderid AS BIGINT)

-- JOIN device_mapping AS device_mapping
--   ON device_mapping.source_concept_code = COALESCE(
--        NULLIF(REGEXP_REPLACE(dbo_qo_classification_de.hicpcscode, '[\\s\\u00A0]+', ''), ''),
--        NULLIF(REGEXP_REPLACE(dbo_qo_classification_de.cpt4code, '[\\s\\u00A0]+', ''), '')
--      )

-- LEFT JOIN _exponent.omop_mapping.source_to_person AS source_to_person
--   ON source_to_person.person_source_value = CONCAT_WS(
--        chr(31),
--        'allscripts_tw',
--        'dbo_person',
--        'id',
--        CAST(dbo_person.id AS BIGINT)
--      )
--  AND source_to_person.active_flag = TRUE

-- LEFT JOIN _exponent.omop_mapping.source_to_provider AS source_to_provider
--   ON source_to_provider.provider_source_value = CONCAT_WS(
--        chr(31),
--        'allscripts_tw',
--        'dbo_provider',
--        'id',
--        CAST(dbo_provider.id AS BIGINT)
--      )
--  AND source_to_provider.active_flag = TRUE

-- LEFT JOIN _exponent.omop_mapping.source_to_visit_occurrence AS source_to_visit_occurrence
--   ON source_to_visit_occurrence.visit_occurrence_source_value = CONCAT_WS(
--        chr(31),
--        'allscripts_tw',
--        'dbo_visit',
--        'id',
--        CAST(dbo_visit.id AS BIGINT)
--      )
--  AND source_to_visit_occurrence.active_flag = TRUE

-- WHERE 1=1
--   -- AND dbo_order_activity.orderstatusde IN (3, 4, 18, 20)
--   -- AND dbo_order_activity_header.etl_load_ts BETWEEN CURRENT_TIMESTAMP() - INTERVAL 14 DAYS AND CURRENT_TIMESTAMP();

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW device_exposure_silver AS
SELECT
  device_exposure_source_value,
  person_id,
  device_concept_id,
  device_exposure_start_date,
  device_exposure_start_datetime,
  device_exposure_end_date,
  device_exposure_end_datetime,
  device_type_concept_id,
  unique_device_id,
  production_id,
  quantity,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  device_source_value,
  device_source_concept_id,
  unit_concept_id,
  unit_source_value,
  unit_source_concept_id,
  source_system
FROM device_exposure_charge;

In [0]:
%sql
MERGE INTO _exponent.omop_silver.device_exposure AS target
USING device_exposure_silver AS source
ON target.device_exposure_source_value = source.device_exposure_source_value

WHEN MATCHED AND NOT (
     target.person_id <=> source.person_id
 AND target.device_concept_id <=> source.device_concept_id
 AND target.device_exposure_start_date <=> source.device_exposure_start_date
 AND target.device_exposure_start_datetime <=> source.device_exposure_start_datetime
 AND target.device_exposure_end_date <=> source.device_exposure_end_date
 AND target.device_exposure_end_datetime <=> source.device_exposure_end_datetime
 AND target.device_type_concept_id <=> source.device_type_concept_id
 AND target.unique_device_id <=> source.unique_device_id
 AND target.production_id <=> source.production_id
 AND target.quantity <=> source.quantity
 AND target.provider_id <=> source.provider_id
 AND target.visit_occurrence_id <=> source.visit_occurrence_id
 AND target.visit_detail_id <=> source.visit_detail_id
 AND target.device_source_value <=> source.device_source_value
 AND target.device_source_concept_id <=> source.device_source_concept_id
 AND target.unit_concept_id <=> source.unit_concept_id
 AND target.unit_source_value <=> source.unit_source_value
 AND target.unit_source_concept_id <=> source.unit_source_concept_id
 AND target.source_system <=> source.source_system
) THEN UPDATE SET
  target.person_id = source.person_id,
  target.device_concept_id = source.device_concept_id,
  target.device_exposure_start_date = source.device_exposure_start_date,
  target.device_exposure_start_datetime = source.device_exposure_start_datetime,
  target.device_exposure_end_date = source.device_exposure_end_date,
  target.device_exposure_end_datetime = source.device_exposure_end_datetime,
  target.device_type_concept_id = source.device_type_concept_id,
  target.unique_device_id = source.unique_device_id,
  target.production_id = source.production_id,
  target.quantity = source.quantity,
  target.provider_id = source.provider_id,
  target.visit_occurrence_id = source.visit_occurrence_id,
  target.visit_detail_id = source.visit_detail_id,
  target.device_source_value = source.device_source_value,
  target.device_source_concept_id = source.device_source_concept_id,
  target.unit_concept_id = source.unit_concept_id,
  target.unit_source_value = source.unit_source_value,
  target.unit_source_concept_id = source.unit_source_concept_id,
  target.source_system = source.source_system,
  target.last_mod_tsp = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN INSERT (
  device_exposure_source_value,
  person_id,
  device_concept_id,
  device_exposure_start_date,
  device_exposure_start_datetime,
  device_exposure_end_date,
  device_exposure_end_datetime,
  device_type_concept_id,
  unique_device_id,
  production_id,
  quantity,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  device_source_value,
  device_source_concept_id,
  unit_concept_id,
  unit_source_value,
  unit_source_concept_id,
  source_system,
  last_mod_tsp
) VALUES (
  source.device_exposure_source_value,
  source.person_id,
  source.device_concept_id,
  source.device_exposure_start_date,
  source.device_exposure_start_datetime,
  source.device_exposure_end_date,
  source.device_exposure_end_datetime,
  source.device_type_concept_id,
  source.unique_device_id,
  source.production_id,
  source.quantity,
  source.provider_id,
  source.visit_occurrence_id,
  source.visit_detail_id,
  source.device_source_value,
  source.device_source_concept_id,
  source.unit_concept_id,
  source.unit_source_value,
  source.unit_source_concept_id,
  source.source_system,
  CURRENT_TIMESTAMP()
);

In [0]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_device_exposure (
  source_system,
  device_exposure_source_value,
  active_flag,
  created_tsp,
  last_mod_tsp,
  merge_id,
  merge_reason
)
SELECT
  source_distinct.source_system,
  source_distinct.device_exposure_source_value,
  TRUE AS active_flag,
  CURRENT_TIMESTAMP() AS created_tsp,
  COALESCE(source_distinct.last_mod_tsp, CURRENT_TIMESTAMP()) AS last_mod_tsp,
  NULL AS merge_id,
  NULL AS merge_reason
FROM (
  SELECT DISTINCT
    source_system,
    device_exposure_source_value,
    last_mod_tsp
  FROM _exponent.omop_silver.device_exposure
  WHERE device_exposure_source_value IS NOT NULL
) source_distinct
LEFT ANTI JOIN _exponent.omop_mapping.source_to_device_exposure existing
  ON source_distinct.device_exposure_source_value = existing.device_exposure_source_value;

In [0]:
%sql
SELECT
    source_to_device_exposure.device_exposure_id,
    device_exposure.device_exposure_source_value,
    device_exposure.person_id,
    device_exposure.device_concept_id,
    device_exposure.device_exposure_start_date,
    device_exposure.device_exposure_start_datetime,
    device_exposure.device_exposure_end_date,
    device_exposure.device_exposure_end_datetime,
    device_exposure.device_type_concept_id,
    device_exposure.unique_device_id,
    device_exposure.production_id,
    device_exposure.quantity,
    device_exposure.provider_id,
    device_exposure.visit_occurrence_id,
    device_exposure.visit_detail_id,
    device_exposure.device_source_value,
    device_exposure.device_source_concept_id,
    device_exposure.unit_concept_id,
    device_exposure.unit_source_value,
    device_exposure.unit_source_concept_id,
    device_exposure.source_system,
    device_exposure.last_mod_tsp
  FROM _exponent.omop_silver.device_exposure
  JOIN _exponent.omop_mapping.source_to_device_exposure
    ON device_exposure.device_exposure_source_value = source_to_device_exposure.device_exposure_source_value
   AND source_to_device_exposure.active_flag = TRUE
  WHERE 1=1
  AND device_exposure.source_system = 'allscripts_tw'

In [0]:
%sql
MERGE INTO _exponent.omop_tw.device_exposure AS target
USING (
  SELECT
    source_to_device_exposure.device_exposure_id,
    device_exposure.device_exposure_source_value,
    device_exposure.person_id,
    device_exposure.device_concept_id,
    device_exposure.device_exposure_start_date,
    device_exposure.device_exposure_start_datetime,
    device_exposure.device_exposure_end_date,
    device_exposure.device_exposure_end_datetime,
    device_exposure.device_type_concept_id,
    device_exposure.unique_device_id,
    device_exposure.production_id,
    device_exposure.quantity,
    device_exposure.provider_id,
    device_exposure.visit_occurrence_id,
    device_exposure.visit_detail_id,
    device_exposure.device_source_value,
    device_exposure.device_source_concept_id,
    device_exposure.unit_concept_id,
    device_exposure.unit_source_value,
    device_exposure.unit_source_concept_id,
    device_exposure.source_system,
    device_exposure.last_mod_tsp
  FROM _exponent.omop_silver.device_exposure AS device_exposure
  JOIN _exponent.omop_mapping.source_to_device_exposure AS source_to_device_exposure
    ON device_exposure.device_exposure_source_value = source_to_device_exposure.device_exposure_source_value
   AND source_to_device_exposure.active_flag = TRUE
  WHERE device_exposure.source_system = 'allscripts_tw'
) AS source
ON target.device_exposure_id = source.device_exposure_id

WHEN MATCHED AND NOT (
     target.person_id <=> source.person_id
 AND target.device_concept_id <=> source.device_concept_id
 AND target.device_exposure_start_date <=> source.device_exposure_start_date
 AND target.device_exposure_start_datetime <=> source.device_exposure_start_datetime
 AND target.device_exposure_end_date <=> source.device_exposure_end_date
 AND target.device_exposure_end_datetime <=> source.device_exposure_end_datetime
 AND target.device_type_concept_id <=> source.device_type_concept_id
 AND target.unique_device_id <=> source.unique_device_id
 AND target.production_id <=> source.production_id
 AND target.quantity <=> source.quantity
 AND target.provider_id <=> source.provider_id
 AND target.visit_occurrence_id <=> source.visit_occurrence_id
 AND target.visit_detail_id <=> source.visit_detail_id
 AND target.device_source_value <=> source.device_source_value
 AND target.device_source_concept_id <=> source.device_source_concept_id
 AND target.unit_concept_id <=> source.unit_concept_id
 AND target.unit_source_value <=> source.unit_source_value
 AND target.unit_source_concept_id <=> source.unit_source_concept_id
) THEN UPDATE SET
  target.person_id = source.person_id,
  target.device_concept_id = source.device_concept_id,
  target.device_exposure_start_date = source.device_exposure_start_date,
  target.device_exposure_start_datetime = source.device_exposure_start_datetime,
  target.device_exposure_end_date = source.device_exposure_end_date,
  target.device_exposure_end_datetime = source.device_exposure_end_datetime,
  target.device_type_concept_id = source.device_type_concept_id,
  target.unique_device_id = source.unique_device_id,
  target.production_id = source.production_id,
  target.quantity = source.quantity,
  target.provider_id = source.provider_id,
  target.visit_occurrence_id = source.visit_occurrence_id,
  target.visit_detail_id = source.visit_detail_id,
  target.device_source_value = source.device_source_value,
  target.device_source_concept_id = source.device_source_concept_id,
  target.unit_concept_id = source.unit_concept_id,
  target.unit_source_value = source.unit_source_value,
  target.unit_source_concept_id = source.unit_source_concept_id

WHEN NOT MATCHED THEN INSERT (
  device_exposure_id,
  person_id,
  device_concept_id,
  device_exposure_start_date,
  device_exposure_start_datetime,
  device_exposure_end_date,
  device_exposure_end_datetime,
  device_type_concept_id,
  unique_device_id,
  production_id,
  quantity,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  device_source_value,
  device_source_concept_id,
  unit_concept_id,
  unit_source_value,
  unit_source_concept_id
) VALUES (
  source.device_exposure_id,
  source.person_id,
  source.device_concept_id,
  source.device_exposure_start_date,
  source.device_exposure_start_datetime,
  source.device_exposure_end_date,
  source.device_exposure_end_datetime,
  source.device_type_concept_id,
  source.unique_device_id,
  source.production_id,
  source.quantity,
  source.provider_id,
  source.visit_occurrence_id,
  source.visit_detail_id,
  source.device_source_value,
  source.device_source_concept_id,
  source.unit_concept_id,
  source.unit_source_value,
  source.unit_source_concept_id
);